In [17]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"

from pathlib import Path

# =============================================================================
# Project directories
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = PROJECT_ROOT / "figures"

SAVE_HTML = True
SHOW_FIGURE = False

In [18]:
clean_dataset_file = RAW_DATA_DIR / "caes_dataset_clean.csv"
df_clean = pd.read_csv(clean_dataset_file, index_col='timestamp')
df_clean.index = pd.to_datetime(df_clean.index)
#
noisy_dataset_file = RAW_DATA_DIR / "caes_dataset_noisy.csv"
df_noisy = pd.read_csv(noisy_dataset_file, index_col='timestamp')
df_noisy.index = pd.to_datetime(df_noisy.index)

# Data Quality Assessment

Before performing operational analysis, the dataset is validated through a set of automated quality checks.

The objective is to identify potential issues commonly found in industrial time-series data:

- missing measurements
- duplicated timestamps
- physically unrealistic values
- inconsistent operating conditions

Although this dataset is synthetic, these checks reproduce the type of validation typically applied to real energy asset monitoring data.

In [19]:
# =============================================================================
# Dataset overview
# =============================================================================
df = df_noisy.copy()
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 17280 entries, 2026-01-01 00:00:00 to 2026-01-02 23:59:50
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   operating_mode       17280 non-null  str    
 1   ambient_temperature  17203 non-null  float64
 2   tank_pressure        17280 non-null  float64
 3   tank_temperature     17231 non-null  float64
 4   compressor_power     17280 non-null  float64
 5   turbine_power        17280 non-null  float64
 6   mass_flow            17280 non-null  float64
dtypes: float64(6), str(1)
memory usage: 1.2 MB


In [20]:
df.describe()

,ambient_temperature,tank_pressure,tank_temperature,compressor_power,turbine_power,mass_flow
count,17203.000000,17280.000000,17231.000000,17280.000000,17280.000000,17280.000000
mean,19.987047,230.542232,28.175961,29.794037,9.432400,0.108311
std,2.117181,38.224833,18.011500,52.292060,21.117285,0.445905
min,17.000000,-153.129091,8.970266,-456.739599,0.000000,-0.614933
25%,17.871460,199.958959,17.236089,0.000000,0.000000,0.000000
50%,19.979273,235.500176,20.549333,0.000000,0.000000,0.000000
75%,22.098735,252.898475,40.552056,0.000000,0.000000,0.178523
max,23.000000,486.386165,61.042771,526.728157,65.303336,0.864459


In [21]:
# =============================================================================
# Missing values analysis
# =============================================================================
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().mean() * 100
    )
})

display(missing_report)

# =============================================================================
# Timestamp validation
# =============================================================================

time_step = (
    df.index.to_series()
    .diff()
    .dropna()
)

display(time_step.value_counts())

expected_frequency = pd.Timedelta(seconds=10)

timestamp_errors = (
    time_step != expected_frequency
).sum()

print(
    f"Timestamp irregularities: {timestamp_errors}"
)

# =============================================================================
# Physical validation rules
# =============================================================================

validation_limits = {

    "tank_pressure": {
        "min": 100,
        "max": 400
    },

    "tank_temperature": {
        "min": -20,
        "max": 100
    },

    "mass_flow": {
        "min": -1,
        "max": 1
    },

    "compressor_power": {
        "min": 0,
        "max": 200
    },

    "turbine_power": {
        "min": 0,
        "max": 100
    }
}
##
def check_physical_limits(
    dataframe,
    limits
):
    """
    Check whether measurements are within
    predefined physical operating ranges.
    """

    results = []

    for variable, bounds in limits.items():

        violations = (
            (dataframe[variable] < bounds["min"]) |
            (dataframe[variable] > bounds["max"])
        ).sum()

        results.append(
            {
                "variable": variable,
                "violations": violations
            }
        )

    return pd.DataFrame(results)
##
quality_report = check_physical_limits(
    df,
    validation_limits
)

display(quality_report)

# =============================================================================
# Operating mode consistency checks
# =============================================================================

invalid_compressor_operation = (
    (df["operating_mode"] != "CHARGING") &
    (df["compressor_power"] > 0)
).sum()


invalid_turbine_operation = (
    (df["operating_mode"] != "DISCHARGING") &
    (df["turbine_power"] > 0)
).sum()


print(
    f"Invalid compressor operation: {invalid_compressor_operation}"
)

print(
    f"Invalid turbine operation: {invalid_turbine_operation}"
)

# =============================================================================
# Quality summary
# =============================================================================

quality_summary = {
    "missing_values": int(df.isnull().sum().sum()),
    "duplicate_timestamps": int(df.index.duplicated().sum()),
    "timestamp_errors": int(timestamp_errors),
    "physical_limit_violations": int(
        quality_report["violations"].sum()
    )
}

display(quality_summary)

,missing_count,missing_percentage
operating_mode,0,0.000000
ambient_temperature,77,0.445602
tank_pressure,0,0.000000
tank_temperature,49,0.283565
compressor_power,0,0.000000
turbine_power,0,0.000000
mass_flow,0,0.000000


timestamp
0 days 00:00:10    17279
Name: count, dtype: int64

Timestamp irregularities: 0


,variable,violations
0,tank_pressure,10
1,tank_temperature,0
2,mass_flow,0
3,compressor_power,10
4,turbine_power,0


Invalid compressor operation: 3
Invalid turbine operation: 0


{'missing_values': 126,
 'duplicate_timestamps': 0,
 'timestamp_errors': 0,
 'physical_limit_violations': 20}

In [22]:
detected_anomalies = []
##
for column in df.columns:

    missing_indexes = df.index[df[column].isna()]

    for idx in missing_indexes:

        detected_anomalies.append(
            {
                "type": "missing_values",
                "column": column,
                "index": idx,
                "value": None,
                "limit": None,
                "detection_method": "missing_check"
            }
        )
##
##===================================
def detect_physical_limit_violations(
    dataframe,
    limits
):

    anomalies = []

    for variable, bounds in limits.items():

        lower = bounds["min"]
        upper = bounds["max"]

        violations = (
            (dataframe[variable] < lower) |
            (dataframe[variable] > upper)
        )

        for idx in dataframe.index[violations]:

            anomalies.append(
                {
                    "type": "physical_limit_violation",
                    "column": variable,
                    "index": idx,
                    "value": dataframe.loc[idx, variable],
                    "limit": (
                        f"{lower} < x < {upper}"
                    ),
                    "detection_method": "physical_limits"
                }
            )

    return anomalies
##
##===================================
detected_anomalies.extend(
    detect_physical_limit_violations(
        df,
        validation_limits
    )
)
##
##===================================
for idx, row in df.iterrows():

    if (
        row["operating_mode"] != "CHARGING"
        and row["compressor_power"] > 0
    ):

        detected_anomalies.append(
            {
                "type": "operating_mode_inconsistency",
                "column": "compressor_power",
                "index": idx,
                "value": row["compressor_power"],
                "limit": "compressor active only during CHARGING",
                "detection_method": "operating_logic"
            }
        )

##
##===================================
for idx, row in df.iterrows():

    if (
        row["operating_mode"] != "DISCHARGING"
        and row["turbine_power"] > 0
    ):

        detected_anomalies.append(
            {
                "type": "operating_mode_inconsistency",
                "column": "turbine_power",
                "index": idx,
                "value": row["turbine_power"],
                "limit": "compressor active only during CHARGING",
                "detection_method": "operating_logic"
            }
        )

anomaly_report = pd.DataFrame(
    detected_anomalies
)


anomaly_report.to_csv(
    "../reports/detected_anomalies.csv",
    index=False
)

In [23]:
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["tank_pressure"],
        name="Pressure"
    )
)


pressure_faults = anomaly_report[
    anomaly_report["column"]=="tank_pressure"
]


fig.add_trace(
    go.Scatter(
        x=pressure_faults["index"],
        y=pressure_faults["value"],
        mode="markers",
        name="Detected anomalies",
        marker=dict(
            size=10
        )
    )
)


fig.update_layout(
    title="Pressure anomaly detection",
    hovermode="x"
)

# ------------------------------------------------------------------
# Save & Display
# ------------------------------------------------------------------
if SAVE_HTML:
    filename = "pressure_anomaly_detection.html"
    fig.write_html(FIGURES_DIR / filename,
                   include_plotlyjs="cdn")
    
if SHOW_FIGURE:
       fig.show()

